# TOMPEI-CMMD preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.tompei_cmmd import TompeiCMMD, raw_imgs_path, raw_imgs_extension, csv_save_path
from utils.preprocessing import abbreviation_mapping, birads_assessment, get_value, laterality, region_mapping

ds = TompeiCMMD()
ds.set_dataset_name("tompei-cmmd")
ds.clinical_data.head()

### Step 1 — drop rows with an exclusion reason, then keep only rows with a known BI-RADS

In [ ]:
ds.clinical_data = ds.clinical_data[ds.clinical_data["exclusion reason"].isna()]
ds.clinical_data = ds.clinical_data[ds.clinical_data["birads"].notna()]
ds.clinical_data["birads"] = ds.clinical_data["birads"].apply(lambda x: int(x))
ds.clinical_data.shape

### Step 2 — map laterality, BI-RADS, and the location fields to harmonized labels

In [ ]:
ds.clinical_data["laterality"] = ds.clinical_data["laterality"].apply(lambda x: get_value(x, laterality) if pd.notna(x) else None)
ds.clinical_data["birads"] = ds.clinical_data["birads"].apply(lambda x: get_value(x, birads_assessment))
for col in ["mass location", "calcification location", "other findings location",
            "mass - additional lesion location", "calcification - additional lesion location"]:
    ds.clinical_data[col] = ds.clinical_data[col].apply(lambda x: get_value(x, region_mapping) if pd.notna(x) else None)
ds.clinical_data["birads"].value_counts(dropna=False)

### Step 3 — lowercase all free-text columns, then expand known abbreviations

In [ ]:
import re

ds.clinical_data = ds.clinical_data.apply(
    lambda col: col.str.lower() if col.dtype == "object" or col.dtype.name == "string" else col
)
pat = re.compile(r"\b(?:" + "|".join(map(re.escape, abbreviation_mapping.keys())) + r")\b")

ds.clinical_data = ds.clinical_data.map(lambda s: pat.sub(lambda m: abbreviation_mapping[m.group(0)], s) if isinstance(s, str) else s)

ds.clinical_data.head()

### Step 4 — keep only patients with exactly 2 images (one CC + one MLO)

In [ ]:
from glob import glob
import os

ds.clinical_data["n_images"] = ds.clinical_data["patient id"].apply(
    lambda pid: len(glob(os.path.join(raw_imgs_path, pid.upper(), "**", f"*{raw_imgs_extension}"), recursive=True))
)
ds.clinical_data["n_images"].value_counts(dropna=False)

In [ ]:
ds.clinical_data = ds.clinical_data[ds.clinical_data["n_images"] == 2]
ds.clinical_data.shape

## Build exam records for one patient

In [ ]:
row = ds.clinical_data.iloc[0]
exams = ds.process_row(row)
exams

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"patients before per-exam processing: {len(ds.clinical_data)} (x2 images each) | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))